# M2: FM100 group comparisons

`PLANScores.md` M2: `comparisons.compare_fm100_feature` (Mann-Whitney U + effect size, subject-pooled scores, same convention `beh/scripts/features.py`'s `compare_shape_feature` uses) across `comparisons.FEATURES` (TES, PES_RG, PES_BY, VKS_MajRad, VKS_MinRad, VKS_Angle) for the four requested pairs: CTR vs PD, HC vs protan, HC vs deutan, protan vs deutan.

`VKS_Angle` is periodic (folds to `[0, 180)`) -- compared with the same folded-Mann-Whitney approximation `beh/`'s `orientation_deg` already uses (valid as long as a group's angles don't straddle the 0/180 wrap point; see `comparisons.py`'s module docstring).

In [1]:
import sys
sys.path.append('../scripts')

import pandas as pd

import comparisons
import loader

df = loader.load_fm100_raw()

pairs = [
    ('CTR', None, 'PD', None),
    ('CTR', None, 'CVD', 'protan'),
    ('CTR', None, 'CVD', 'deutan'),
    ('CVD', 'protan', 'CVD', 'deutan'),
]
pair_labels = ['CTR vs PD', 'HC vs protan', 'HC vs deutan', 'protan vs deutan']

In [2]:
rows = []
for (g1, sg1, g2, sg2), label in zip(pairs, pair_labels):
    for feature in comparisons.FEATURES:
        result = comparisons.compare_fm100_feature(df, feature, group1=g1, subgroup1=sg1, group2=g2, subgroup2=sg2)
        rows.append({'comparison': label, **result})
results = pd.DataFrame(rows)
results

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
0,CTR vs PD,TES,21.5,0.016728,-0.658730,0.170635,21,6
1,CTR vs PD,PES_RG,17.5,0.008648,-0.722222,0.138889,21,6
2,CTR vs PD,PES_BY,22.0,0.018159,-0.650794,0.174603,21,6
3,CTR vs PD,VKS_MajRad,23.0,0.018148,-0.634921,0.182540,21,6
4,CTR vs PD,VKS_MinRad,20.0,0.010223,-0.682540,0.158730,21,6
5,CTR vs PD,VKS_Angle,59.0,0.842472,-0.063492,0.468254,21,6
6,HC vs protan,TES,3.0,0.000085,-0.964286,0.017857,21,8
7,HC vs protan,PES_RG,0.0,0.000046,-1.000000,0.000000,21,8
8,HC vs protan,PES_BY,33.0,0.013722,-0.607143,0.196429,21,8
9,HC vs protan,VKS_MajRad,3.0,0.000003,-0.964286,0.017857,21,8


## Just the significant rows (p < 0.05, uncorrected -- matching every other group-comparison battery in this project, e.g. `beh/`'s M1)

In [3]:
results[results['p_value'] < 0.05].sort_values('p_value')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
16,HC vs deutan,VKS_MinRad,3.0,0.000003,-0.964286,0.017857,21,8
9,HC vs protan,VKS_MajRad,3.0,0.000003,-0.964286,0.017857,21,8
15,HC vs deutan,VKS_MajRad,4.0,0.000006,-0.952381,0.023810,21,8
7,HC vs protan,PES_RG,0.0,0.000046,-1.000000,0.000000,21,8
13,HC vs deutan,PES_RG,0.0,0.000046,-1.000000,0.000000,21,8
10,HC vs protan,VKS_MinRad,10.0,0.000063,-0.880952,0.059524,21,8
6,HC vs protan,TES,3.0,0.000085,-0.964286,0.017857,21,8
12,HC vs deutan,TES,3.5,0.000094,-0.958333,0.020833,21,8
14,HC vs deutan,PES_BY,14.0,0.000695,-0.833333,0.083333,21,8
1,CTR vs PD,PES_RG,17.5,0.008648,-0.722222,0.138889,21,6


## Holm-corrected, per pair (M3)

`comparisons.correct_multiple_comparisons` (Holm, the family-wise-error
default), applied once per comparison (6 features = one family), matching
how `ssvepBeh/`'s own correlation battery is corrected. Family-wise error
control is more conservative than the uncorrected view above -- some rows
significant there don't survive here.

In [4]:
corrected = pd.concat([comparisons.correct_multiple_comparisons(results[results['comparison'] == label]) for label in pair_labels])
corrected[corrected['significant']].sort_values('p_corrected')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2,p_corrected,significant
9,HC vs protan,VKS_MajRad,3.0,0.000003,-0.964286,0.017857,21,8,0.000020,True
16,HC vs deutan,VKS_MinRad,3.0,0.000003,-0.964286,0.017857,21,8,0.000020,True
15,HC vs deutan,VKS_MajRad,4.0,0.000006,-0.952381,0.023810,21,8,0.000028,True
13,HC vs deutan,PES_RG,0.0,0.000046,-1.000000,0.000000,21,8,0.000184,True
7,HC vs protan,PES_RG,0.0,0.000046,-1.000000,0.000000,21,8,0.000230,True
10,HC vs protan,VKS_MinRad,10.0,0.000063,-0.880952,0.059524,21,8,0.000253,True
6,HC vs protan,TES,3.0,0.000085,-0.964286,0.017857,21,8,0.000255,True
12,HC vs deutan,TES,3.5,0.000094,-0.958333,0.020833,21,8,0.000282,True
14,HC vs deutan,PES_BY,14.0,0.000695,-0.833333,0.083333,21,8,0.001389,True
8,HC vs protan,PES_BY,33.0,0.013722,-0.607143,0.196429,21,8,0.026847,True
